# Tempo-Core: fine-tune Qwen3-1.7B on Tempo's checked answers

LoRA SFT on `sft/` (answers that passed Tempo's checks), then DPO on `pairs/` (the answer that
passed against a draft that failed), then merge, convert to GGUF and quantize to **Q4_K_M** for
Ollama or llama.cpp on an ordinary CPU. The data comes from `tempo-server train prepare`; the
code is the training kit that came with it (`tempo_trainkit.py`).

**Before you run** (right panel): *Session options → Accelerator → GPU T4 x2* and *Internet →
On* (needs a phone-verified Kaggle account); *Add Input → Datasets → Your Work →* the dataset
you uploaded. Then **Save Version → Save & Run All (Commit)**: it runs in the background.

**Time** (estimates until measured; the report records the real times): about 1 hour of SFT
for 5,000 rows and 2 epochs, about 1 hour of DPO for 2,000 pairs, 10-15 minutes for the GGUF
step. Kaggle stops a session at 12 hours: the notebook saves a checkpoint 20 minutes before
its time budget and stops. To resume: *Add Input → Notebook Output →* the stopped version, and
run again.

**Output** (`/kaggle/working/tempo-core/`): `tempo-core-q4_k_m.gguf`, `Modelfile`,
`report.json`, `REPORT.md`. Download the output, then on your computer:
`tempo-server models import <zip>` → `tempo-server models compare` → `tempo-server models
promote`. See docs/TRAINING.md.

In [ ]:
# Settings
import os

# Apache-2.0. The second family: "ibm-granite/granite-3.3-2b-instruct"
BASE_MODEL = "Qwen/Qwen3-1.7B"
TIME_BUDGET_HOURS = 11  # Kaggle stops a session at 12 hours; this leaves time to save
os.environ.setdefault("TEMPO_TIME_BUDGET_H", str(TIME_BUDGET_HOURS))

## 1. Load the pack and the training kit

In [ ]:
# Find the Tempo pack among the inputs (Kaggle unpacks the uploaded zip; a zip still works),
# then load the training kit that came with it.
import glob
import os
import sys
import zipfile
from pathlib import Path

INPUT = Path(os.environ.get("TEMPO_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("TEMPO_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
found = sorted(glob.glob(str(INPUT / "**" / "tempo-pack.json"), recursive=True))
if found:
    PACK = Path(found[-1]).parent
else:
    PACK = None
    for archive in sorted(glob.glob(str(INPUT / "**" / "*.zip"), recursive=True)):
        with zipfile.ZipFile(archive) as z:
            if "tempo-pack.json" in z.namelist():
                PACK = WORK / "pack"
                z.extractall(PACK)
                break
if PACK is None:
    raise SystemExit(
        "No Tempo training pack found. Right panel -> Add Input -> Datasets -> Your Work -> the "
        "dataset you uploaded (made by `tempo-server train prepare`)."
    )
sys.path.insert(0, str(PACK))
import tempo_trainkit as tk  # noqa: E402

print(f"Pack: {PACK} (training kit version {tk.KIT_VERSION})")
clock = tk.Clock()

## 2. GPU check and packages

In [ ]:
hardware = tk.gpu_check(min_gpus=1)
tk.install(tk.CORE_PACKAGES)

## 3. Data, settings, and an earlier run to resume from

In [ ]:
manifest = tk.load_pack(PACK)
tk.resume_from_inputs(INPUT, WORK)
state = tk.State(WORK)
cfg = tk.CoreConfig.for_run()
cfg.base = BASE_MODEL
print(cfg)

## 4. SFT (LoRA) on checked answers

In [ ]:
sft_metrics = tk.sft(PACK, WORK, cfg, clock, state)

## 5. DPO on preference pairs (same LoRA)

In [ ]:
dpo_metrics = tk.dpo(PACK, WORK, cfg, clock, state) if state.done("sft") else {}

## 6. Merge, convert to GGUF, quantize to Q4_K_M

In [ ]:
files = tk.core_outputs(WORK, cfg, state) if state.done("dpo") else {}

## 7. Report

In [ ]:
complete = state.done("gguf")
report = {
    "model": "Tempo-Core",
    "status": "complete" if complete else "incomplete",
    "run_id": tk.run_id(),
    "hardware": ", ".join(hardware["gpus"]) or hardware["cpu"],
    "base": cfg.base + (" (tiny random copy: dry run)" if cfg.tiny else ""),
    "base_licence": tk.CORE_BASES[cfg.base]["licence"],
    "base_source": tk.CORE_BASES[cfg.base]["source"],
    "ollama_base": tk.CORE_BASES[cfg.base]["ollama_base"],
    "dry_run": cfg.tiny,
    "settings": {k: v for k, v in vars(cfg).items()},
    "stages": {k: state.get(k) for k in ("sft", "dpo", "gguf") if state.get(k)},
    "heldout": {
        "sft_loss_before": state.get("sft").get("heldout_loss_before"),
        "sft_loss_after": state.get("sft").get("heldout_loss_after"),
        "dpo_reward_accuracy": state.get("dpo").get("heldout_reward_accuracy"),
        "dpo_reward_margin": state.get("dpo").get("heldout_reward_margin"),
    },
    "data": manifest,
    "files": files,
    "seconds": round(clock.elapsed()),
}
tk.write_report(WORK / "tempo-core", report)